<!-- <h1 style="color:#004080; font-size:2em; margin-bottom:0.3em;"> Comparing NOAA AORC vs USGS CONUS404 Across Headwater Watersheds</h1>
<h4 style="color:gray; font-weight:normal;">
 -->
## Comparing NOAA AORC vs USGS CONUS404 Across Headwater Watersheds

This use case explores how two high-resolution meteorological forcing datasets: the USGS/NCAR CONUS404 reanalysis and NOAA's Analysis of Record for Calibration (AORC). By examining their precipitation outputs over several diverse headwater watersheds, we aim to assess the consistency between these data sources and understand the differences in forcing data that could lead to different streamflow predictions. Also determining whether the pattern of differences is consistent across regions, so guiding researchers on how to deal with multi-source meteorological data in modeling scenarios.

### Use case overview

**Authors:**
- Irene Garousi-Nejad, [igarousi@cuahsi.org](mailto:igarousi@cuahsi.org)
- Anthony Castronova, [acastronova@cuahsi.org](mailto:acastronova@cuahsi.org)
- Abner Bogan, [abogan@cuahsi.org](mailto:abogan@cuahsi.org)

**Last Modified:** 8/1/25

**Content:**
- Preparation
- Study Watersheds and Selected Wet vs. Drought Periods
- Explore and Access CONUS404
- Explore and Access AORC
- Aligning Gridded Datasets with Watershed Boundaries
- Investigating Biases, Variability, and Correlations

**Software Requirements:** This notebook has been tested using Python v3.10 using the snapshot of library versions defined in the [Preparation](Preparation) section.

**Supporting Files and Dependencies:**
- `setup_conus404_env.sh`: shell script executing commands to install required dependanices to run this notebook and register this environment as a Jupyter kernel for running in JupyterHub.
- `utils.py`: Utility script containing functinos for processing the gridded weather datasets and plotting metrics of interest.
- Memory requirements: 15 GB, large machine

**References:**
- https://hytest-org.github.io/hytest/dataset_access/conus404_explore.html
- Fuchs, B., Wood, D., Ebbeka, D., and Bergantino, A. (2015). "From Too Much to Too Little: How the central U.S. drought of 2012 evolved out of one of the most devastating floods on record in 2011".
- Rippey, B. R. (2015), The U.S. drought of 2012, Weather and Climate Extremes, 10, 57–64, https://doi.org/10.1016/j.wace.2015.10.004.
- Rasmussen, R.M., Chen, F., Liu, C., Ikeda, K., Prein, A., Kim, J., Schneider, T., Dai, A., Gochis, D., Dugger, A., Zhang, Y., Jaye, A., Dudhia, J., He, C., Harrold, M., Xue, L., Chen, S., Newman, A., Dougherty, E., Abolafia-Rozenzweig, R., Lybarger, N., Viger, R., Dunne, K., Rasmussen, K., and Miguez-Macho, G., 2023, CONUS404: Four-kilometer long-term regional hydroclimate reanalysis over the conterminous United States (ver. 2.0, December 2023): U.S. Geological Survey data release, https://doi.org/10.5066/P9PHPK4F.

### Preparation

##### Environment and Jupyter kernel configuration

This notebook was developed in the *CIROH 2i2c JupyterHub* environment using the large base image. To ensure the smoothest experience when running the notebooks on *CIROH 2i2c JupyterHub* or other available JupyterHub environments, including *CUAHSI JupyterHub*, you need to first **create a custom conda environment** in your home directory to ensure all required Python modules are available.

To do this, run the command `bash ./setup_conus404_env.sh` in the terminal in the directory of the notebook to create/activate the Conda environment and Jupyter Kernel. After it's finished, restart the Jupyter kernel and select the Python kernel called `Python [conda env:conus404-env]` to begin working with the new environment. Note that this setup will install the following libraries:
- geopandas=1.1.1
- matplotlib=3.10.3
- xarray=2025.6.1
- rioxarray=0.19.0
- s3fs=2025.7.0
- contextily=1.6.2
- zarr=2.17.2
- exactextract=0.2.0
- dask=2025.7.0
- ipykernel=6.29
- conda-tree=1.1.1

#### Import Required Python Libraries

We use tools such as `xarray`, `dask`, and `geopandas` to access and process data, and libraries like `folium` and `shapely` to create and visualize maps within this Jupyter notebook.

In [1]:
import os
import pyproj

os.environ['USE_PYGEOS'] = '0'

import fsspec
import xarray as xr
import hvplot.xarray
import intake
import cartopy.crs as ccrs
import metpy
import geopandas as gpd
import folium
import sys
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from shapely.geometry import LineString, MultiLineString
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import pandas as pd
import rioxarray
import numpy as np
import utils


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/srv/conda/envs/notebook/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/srv/conda/envs/notebook/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/srv/conda/envs/notebook/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/srv/

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/srv/conda/envs/notebook/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/srv/conda/envs/notebook/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/srv/conda/envs/notebook/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/srv/

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/srv/conda/envs/notebook/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/srv/conda/envs/notebook/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/srv/conda/envs/notebook/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/srv/

...


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/srv/conda/envs/notebook/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/srv/conda/envs/notebook/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/srv/conda/envs/notebook/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/srv/

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/srv/conda/envs/notebook/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/srv/conda/envs/notebook/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/srv/conda/envs/notebook/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/srv/

AttributeError: _ARRAY_API not found

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

##### Notes on `Dask` setup

In this notebook, we use `dask` to accelerate access and analysis of both datasets. To manage parallel computation and visualize progress of long-running tasks, we initialize a Dask “cluster,” which defines how many workers are used and how much computing power each worker has. By default, we use `client = Client(cluster)`, which connects our notebook to the cluster. When this client is created, an output like `Client: 'tcp://127.0.0.1:34153' processes=1 threads=16, memory=15.00 GiB` is printed. Here's what it means:
- **'tcp://127.0.0.1:34153'**: This is the address of the scheduler (your local machine, in this case), which coordinates work across Dask workers.
- **processes=1**: Only one Dask process is running, meaning all work will be handled in a single process.
- **threads=16**: That single process is using 16 threads for parallel execution—typically one per CPU core.
- **memory=15.00 GiB**: The process has 15 GiB of memory available for computation and caching.

This default setup is appropriate for many exploratory workflows on laptops or single-node systems. For larger datasets or more intensive analysis, this cluster can be scaled up accordingly.

In [ ]:
import os
from dask.distributed import Client, LocalCluster

cluster = LocalCluster(threads_per_worker=os.cpu_count())
client = Client(cluster)   #Client(n_workers=8, memory_limit='10GB') #Large Machine
print(client)

<br></br>
<div style="display: flex; align-items: center; margin-bottom: 0.3em;">
  <div style="display: flex; align-items: center;">
    <span style="color:#004080; font-size: 1.7em; font-weight: bold;">Study Watersheds and Selected Wet vs. Drought Periods</span>
  </div>
</div>
<section style="max-width: 800px; line-height: 1.6; font-family: Arial, sans-serif; font-size: 1em; color: #333;">
  <p>
    We focus on three headwater-scale watersheds (two HUC-10 levels and one Huc-12 level) in different hydro-climatic regions of the United States. For each basin, we identify a period that includes an extremely wet year followed by an extremely dry (drought) year. This allows us to test each dataset's representation of climate extremes shifts:
  </p>
  <ul>
    <li>
      <strong>Cottonwood Canyon (Utah):</strong> Water Year 2011 as a wet year, followed by Water Year 2012 as a drought year. In 2011, Utah's Wasatch Mountains saw exceptional snowpack and flooding, whereas 2012 was marked by severe dryness and below normal snow.
      (<a href="https://www.drought.gov" target="_blank">drought.gov</a>, and Rippey, 2015).
    </li>    
    <li>
      <strong>Tuolumne River Headwaters (California):</strong> Water Year 2011 (record snow and runoff in the Sierra Nevada) followed by Water Year 2012 (onset of a major drought in California). 2011 brought heavy precipitation across California, but by 2012 the state experienced drought.
      (<a href="https://www.drought.gov" target="_blank">drought.gov</a>, and Rippey, 2015).
    </li>    
    <li>
      <strong>Ottauquechee River near West Bridgewater (Vermont):</strong> Calendar Year 2011 (extreme wet conditions resulting in the floods brought by Tropical Storm Irene) followed by 2012 (drier conditions and regional drought). Vermont's 2011 saw historic flooding (Irene in August 2011), whereas 2012 had significantly lower rainfall and even moderate drought in parts of New England.
      (<a href="https://www.drought.gov" target="_blank">drought.gov</a>, and Rippey, 2015).
    </li>
  </ul>
  <p>
    <strong>Why 2011–2012?</strong> Notably, 2011 was extraordinarily wet in many parts of the U.S. (with epic floods in the spring and tropical-driven rainfall later), and 2012 brought one of the most extensive droughts on record 
    (<a href="https://www.drought.gov" target="_blank">drought.gov</a>). By examining this period in three different watersheds, we test whether AORC and CONUS404 capture the same pattern of extreme wetting and drying consistently across regions. If both datasets agree closely in all cases, it builds confidence in their reliability; if they diverge, understanding those differences is key to deciding which dataset to use for hydrologic applications in each region.
  </p>
</section>


The maps below show the three study watersheds along with their main drainage rivers, highlighting the general flow direction.

In [ ]:
from utils import plot_watersheds_with_flowlines

watershed_files = {
    "Tuolumne River": {
        "watershed": "./watersheds/TuolumneRiver.shp",
        "flowline": "./watersheds/TuolumneRiver_flowline.shp"
    },
    "Cottonwood Canyon": {
        "watershed": "./watersheds/CottonwoodCreek.shp",
        "flowline": "./watersheds/CottonwoodCreek_flowline.shp"
    },
    "Ottauquechee River": {
        "watershed": "./watersheds/OttauquecheeRiver.shp",
        "flowline": "./watersheds/OttauquecheeRiver_flowline.shp"
    }
}

plot_watersheds_with_flowlines(watershed_files)


<div style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-size: 1.05em; color: #333; background-color: #f9f9f9; border-left: 4px solid #004080; padding: 16px; margin-top: 20px; line-height: 1.6;">
  <ul style="margin-left: 20px; padding-left: 0;">
    <li><strong>Tuolumne River Watershed (California):</strong> Water flows from southeast to west across a mountainous region within the <em>Sierra Nevada hydro-climatic zone</em>, characterized by snowmelt-driven hydrology.</li>
    <li><strong>Cottonwood Canyon (Utah):</strong> A smaller <em>HUC-12 basin</em> where water flows from east to west through an <em>arid to semi-arid region</em> of the Intermountain West.</li>
    <li><strong>Ottauquechee River Watershed (Vermont):</strong> Water flows from west to east across a <em>humid continental climate</em>, influenced by seasonal precipitation and snowmelt.</li>
  </ul>
  <p style="margin-bottom: 0;">Unlike Cottonwood Canyon, the Tuolumne and Ottauquechee watersheds are both <strong>HUC-10 basins</strong>, reflecting their larger spatial extent and more complex drainage networks.</p>
</div>


Define the period of interest:

In [ ]:
start_date='2011-01-01'
end_date='2012-12-31'

<br></br>
<div style="display: flex; align-items: center; margin-bottom: 0.3em;">
  <div style="display: flex; align-items: center;">
    <img src="https://tse2.mm.bing.net/th/id/OIP.-02i2agKqmB5pAPhSRgvegAAAA?r=0&rs=1&pid=ImgDetMain&o=7&rm=3" 
         alt="NOAA Logo" 
         style="height: 50px; margin-right: 16px;">
    <span style="color:#004080; font-size: 24px; font-weight: bold;">Explore and Access CONUS404</span>

  </div>
</div>

<section style="max-width: 800px; line-height: 1.6; font-family: Arial, sans-serif; font-size: 1em; color: #333;">
    <p>
      <strong>CONUS404</strong> is a high-resolution hydro-climate dataset covering over 40 years (water years 1980–2022) of the CONtiguous United States (CONUS) at 4-km resolution, which is developed using Weather Research and Forecasting (WRF) model simulations by the National Center for Atmospheric Research (NCAR) in collaboration with the U.S. Geological Survey (USGS). The Zarr-based version, hosted through the <strong>HyTEST</strong> (Hydro-Terrestrial Earth System Testbed) initiative (a joint effort by USGS and NCAR) is openly accessible via the <a href="https://hytest-org.github.io/hytest/dataset_access/CONUS404_CHANGELOG.html" target="_blank">HyTEST Intake Data Catalog</a>. By leveraging the cloud-native Zarr format of this dataset, users can interact with data on demand using scalable tools like Dask, without downloading entire datasets or requiring credentials. This makes CONUS404 easy to use across diverse computing platforms and well-suited for large-scale, reproducible research workflows.
    </p>
</section>

Navigate the HyTEST's intake catalog to select and open a dataset. In this Jupyter notebook, we will use the `conus404-catalog`. 

In [ ]:
# open the hytest data intake catalog
hytest_cat = intake.open_catalog("https://raw.githubusercontent.com/hytest-org/hytest/main/dataset_catalog/hytest_intake_catalog.yml")
list(hytest_cat)

<div style="font-family: Arial, sans-serif; font-size: 16px; line-height: 1.6; color: #333;">
    
  <p>
    Open the <code>conus404-catalog</code> sub-catalog to see available datasets. A sub-catalog may contain a set of datasets for a particular use case. 
    For example, the CONUS404 datasets (at different time steps and storage locations) are stored in their own sub-catalog  
    (<a href="https://hytest-org.github.io/hytest/dataset_access/CONUS404_ACCESS.html" target="_blank">learn more</a>).
  </p>
</div>


In [ ]:
cat = hytest_cat['conus404-catalog']
list(cat)

<br>
  <p>
    In this Jupyter Notebook, we will work with hourly datasets on the Open Storage Network  (OSN) pod, which does not need any credentials to access the data. 
    Also, we will focus on the bias-adjusted dataset for the purpose of this notebook 
    (<code>conus404-hourly-ba-osn</code>), which has been revised to include 43 years of data at 1-kilometer grid spacing. This is a dataset of historical conditions (water years 1980–2022, October 1, 1979–September 30, 2022) and has sufficient temporal and spatial detail to make it appropriate for forcing hydrological models and conducting meteorological analyses 
    (<a href="https://www.sciencebase.gov/catalog/item/64f77cad34ed30c20544c18" target="_blank">view source</a>).
  </p>
</br>

Select the dataset and preview its metadata

In [ ]:
dataset = 'conus404-hourly-ba-osn' 
print(f"Reading {dataset} metadata...", end='')
ds_conus404 = cat[dataset].to_dask().metpy.parse_cf()
ds_conus404

Notice that this loaded very fast. That’s because it performed a “lazy” load of the data, i.e. only the metadata was loaded. Data values will not be accessed until computations are performed.

In [ ]:
print(f'Total Size of .........: {ds_conus404.nbytes/1e12:.1f} TB')
print(f'Size Loaded into Memory: {sys.getsizeof(ds_conus404)} bytes')

Check the precipitation variable

In [ ]:
ds_conus404.RAINRATE

Check the spatial resolution of the gridded data along the y dimension

In [ ]:
ds_conus404.y[1]-ds_conus404.y[0]

<p>
  <strong>So far</strong>, we have demonstrated how to access this high-value dataset and explore its dimensions, coordinates, variables, and metadata.
  In the next section, we'll shift focus to accessing the <strong>AORC</strong> dataset. 
  Once both datasets are prepared, we will return to <code>ds_conus404</code> and load the actual data into memory for further analysis.
</p>


<div style="display: flex; align-items: center; margin-bottom: 0.3em;">
  <div style="display: flex; align-items: center;">
    <img src="https://www.noaa.gov/sites/default/files/2022-03/noaa_emblem_logo-2022.png" 
         alt="NOAA Logo" 
         style="height: 50px; margin-right: 16px;">
    <span style="color:#004080; font-size: 24px; font-weight: bold;">Explore and Access AORC</span>

  </div>
</div>

<section style="max-width: 800px; line-height: 1.6; font-family: Arial, sans-serif; font-size: 1em; color: #333;">
      <p> <strong>AORC</strong>, developed by NOAA’s National Weather Service Office of Water Prediction, provides high-resolution (1-km), hourly gridded meteorological data across the conterminous United States (CONUS), Alaska, and neighboring regions that contribute to U.S. watersheds. AORC integrates observations and model estimates to generate the best available inputs for hydrologic modeling from 1979 to the near-present. In this notebook, we will work with version 1.1 of AORC, which is publicly available as part of the <a href="https://registry.opendata.aws/noaa-nwm-retrospective/" target="_blank">NOAA National Water Model v3.0 Retrospective archive</a> on AWS. These data are also accessible in <strong>Zarr</strong> format, allowing efficient, cloud-native access and scalable analysis using tools like <code>xarray</code>.
    </p>
</section>


Define a few parameters for accessing the specific variable that we are interested in.

In [ ]:
bucket_url = 's3://noaa-nwm-retrospective-3-0-pds'
region = 'CONUS'
variable = 'precip'

In [ ]:
# NOTE TO SELF:
# Previously, we used this method (see below) to load Zarr stores, but it no longer works 
# after updating to newer versions of Zarr (v2.16+ and v3 beta). The recent API 
# changes do not support using FSMap objects directly when opening consolidated Zarr datasets.
# 
# Recommended solution:
# Let xarray manage the S3 filesystem internally by passing the `s3://` URL 
# along with appropriate `storage_options` instead of using `fsspec.get_mapper`.

# Example of the old approach (now deprecated for new Zarr versions):
# s3path = f"{bucket_url}/{region}/zarr/forcing/{variable}.zarr"
# ds_aorc = xr.open_zarr(fsspec.get_mapper(s3path, anon=True), consolidated=True)

In [ ]:
print(f"Reading AORC metadata...", end='')
ds_aorc = xr.open_zarr(
    store='s3://noaa-nwm-retrospective-3-0-pds/CONUS/zarr/forcing/precip.zarr',
    consolidated=True,
    storage_options={"anon": True}
)

ds_aorc

Preview the size

In [ ]:
print(f'Total Size of .........: {ds_aorc.nbytes/1e12:.1f} TB')
print(f'Size Loaded into Memory: {sys.getsizeof(ds_aorc)} bytes')

View the precipitation data

In [ ]:
ds_aorc.RAINRATE

<br></br>
<div style="display: flex; align-items: center; margin-bottom: 0.3em;">
  <div style="display: flex; align-items: center;">
    <span style="color:#004080; font-size: 1.7em; font-weight: bold;">Aligning Gridded Datasets with Watershed Boundaries</span>
  </div>
</div>

To conduct meaningful watershed-scale analysis, it is important to ensure that both gridded datasets are properly aligned with the boundaries of the three selected watersheds. This process involves several important steps: first, inspecting and comparing the coordinate reference systems (CRS) used by each dataset; second, verifying the spatial domain and resolution to ensure consistency in coverage; and finally, clipping each dataset to the watershed extent. By harmonizing spatial properties, we can ensure that comparisons between datasets are accurate and representative of the same geographic area. The steps below walk through this alignment process.

Preview the domain boundaries of both gridded dataset

In [ ]:
print("\n Domain Boundaries for CONUS404 Dataset")
print("------------------------------------------")
print(f"X Range: [{ds_conus404.x.min().item():.2f}, {ds_conus404.x.max().item():.2f}]")
print(f"Y Range: [{ds_conus404.y.min().item():.2f}, {ds_conus404.y.max().item():.2f}]")

print("\n Domain Boundaries for AORC Dataset")
print("--------------------------------------")
print(f"X Range: [{ds_aorc.x.min().item():.2f}, {ds_aorc.x.max().item():.2f}]")
print(f"Y Range: [{ds_aorc.y.min().item():.2f}, {ds_aorc.y.max().item():.2f}]")


Display the Coordinate Reference System (CRS) used by each dataset

In [ ]:
print("CRS (WKT) for CONUS404 dataset:\n-------------------------------")
print(ds_conus404.crs.attrs['crs_wkt'])

print("\nCRS (ESRI PE String) for AORC dataset:\n--------------------------------------")
print(ds_aorc.crs.attrs['esri_pe_string'])


Although both datasets already contain spatial coordinate information and use the `Lambert Conformal Conic` projection, their coordinate reference systems (CRS) differ slightly in format. One specified as a WKT string (`crs_wkt`) and the other as an ESRI PE string (`esri_pe_string`). These subtle differences can lead to compatibility issues when performing spatial operations such as `clip()` or `reproject()` with `rioxarray`.

To ensure consistent and reliable behavior, it is best practice to normalize the CRS using `pyproj.CRS.from_wkt()` or `pyproj.CRS.from_string()` and then explicitly apply it to the dataset using `ds.rio.write_crs(...)`. This step is important even if the dataset includes a `spatial_ref` coordinate, because many spatial operations in `rioxarray` rely on the presence of a properly assigned CRS via `.rio.write_crs(...)` to function correctly.

In [ ]:
# Normalize CRS from WKT (CONUS404)
crs_conus = pyproj.CRS.from_wkt(ds_conus404.crs.attrs['crs_wkt'])

# Normalize CRS from ESRI string (AORC)
crs_aorc = pyproj.CRS.from_wkt(ds_aorc.crs.attrs['esri_pe_string'])

# Confirm equality
print(crs_conus == crs_aorc)  # → True if both are equivalent

# Apply CRS to both datasets (use .rio.write_crs if you already have spatial coords)
ds_conus404.rio.write_crs(crs_conus, inplace=True);
ds_aorc.rio.write_crs(crs_aorc, inplace=True);


Print the compact text format for describing CRS used by the PROJ library.

In [ ]:
print(f'CONUS404 CRS:\n-----\n{ds_conus404.rio.crs.to_proj4()}')
print(f'AORC CRS:\n-----\n{ds_aorc.rio.crs.to_proj4()}')

Next, we need to ensure that the CRS used by the AORC and CONUS404 datasets matches the coordinate reference system of the watershed shapefiles. Since these coordinate systems differ, we will need to reproject the watershed geometries or the datasets accordingly before proceeding with spatial operations.

In [ ]:
watershed_paths = {
    "Tuolumne River": "./watersheds/TuolumneRiver.shp",
    "Cottonwood Canyon": "./watersheds/CottonwoodCreek.shp",
    "Ottauquechee River": "./watersheds/OttauquecheeRiver.shp"
}

# Read, print original CRS, reproject, and print updated CRS
gdfs = {}
for name, path in watershed_paths.items():
    gdf = gpd.read_file(path)
    print(f"\nOriginal CRS for {name}:\n-----\n{gdf.crs.to_wkt()}")

    # convert the shapefile into the coordinate system of the xarray datasets
    gdf_proj = gdf.to_crs(ds_aorc.rio.crs)
    print(f"Reprojected CRS for {name}:\n-----\n{gdf_proj.crs.to_wkt()}")
    # we used .to_wkt() instead of .to_proj4() for debugging or comparing purposes
    
    gdfs[name] = gdf_proj  # Store for later use if needed

### TODO: Testing the process on one watershed. We need to expand it to include the other two. It is better to have the following steps in a reusable function in our utils.py and apply it to all three watersheds in a single code cell.

Let's clip both gridded datasets to the spatial extent of the selected watersheds and the defined time period. This can be accomplished using `rioxarray`'s `clip()` method.

In [ ]:
gdfs.keys()

In [ ]:
# Call the function to clip datasets to all watersheds
clipped_watershed_datasets = utils.clip_watershed_data(
    gdfs=gdfs,
    ds_conus404=ds_conus404,
    ds_aorc=ds_aorc,
    start_date=start_date,
    end_date=end_date
)

In [ ]:
clipped_watershed_datasets

Our datasets are still lazily loaded using `Dask`, especially since we used `from_disk=True` during clipping. Below, we print the estimated total size of the data that would be loaded into memory after clipping and slicing if we were to trigger computation later using `.load()` or `.compute()`.

In [ ]:
# display size of metadata from one watershed as an example
print(f'{clipped_watershed_datasets["Cottonwood Canyon"]["conus404"].nbytes/1e6:.1f} MB')
print(f'{clipped_watershed_datasets["Cottonwood Canyon"]["aorc"].nbytes/1e6:.1f} MB')

<p>
  When working with large datasets using tools like <code>Zarr</code>, <code>xarray</code>, and <code>Dask</code>, it is important to understand how <strong>chunks</strong> work—especially before triggering computations with <code>.load()</code> or <code>.compute()</code>. A <strong>chunk</strong> is a small block of data representing a subset of the entire array, stored and processed independently. Managing chunk size is crucial because if chunks are too large, they can exceed your available memory and cause crashes or significant slowdowns. On the other hand, if chunks are too small, the overhead of managing many tasks can also reduce overall performance.
</p>

<p>
  Ideally, chunk shapes should align with how you plan to access or analyze your data. For instance, if your typical workflow involves slicing along the <code>time</code> dimension, but your chunks are stored as long time-series per location (e.g., <code>time: 1000</code>), you may experience degraded performance. Rechunking to smaller time windows (e.g., <code>time: 50</code>) can significantly improve computational efficiency.
</p>

<p>
  You should consider rechunking if your computations feel very slow, if the current chunking structure does not align well with your analysis patterns, or if you get warnings or memory-related errors when calling <code>.load()</code> or <code>.compute()</code>. Understanding the role of chunking can help you make better use of available resources and improve the speed and memory usage in your workflows.
</p>


<h4>What Do “Chunks” Mean in Each Tool?</h4>
<ul>
  <li><strong>Zarr:</strong> Stores data in chunked format on disk.</li>
  <li><strong>xarray (with Dask):</strong> Operates on data chunk-by-chunk instead of loading everything into memory.</li>
  <li><strong>Dask:</strong> Treats each chunk as a task in a computation graph, enabling parallel execution and memory efficiency.</li>
</ul>


Check the chunks

In [ ]:
# display the chunks from one watershed as an example
print("CONUS404 Chunk Structure:")
print("-" * 40)
print(clipped_watershed_datasets["Cottonwood Canyon"]["conus404"].chunks)
print("\nAORC (Clipped) Chunk Structure:")
print("-" * 40)
print(clipped_watershed_datasets["Cottonwood Canyon"]["aorc"].chunks)

Note that you can rechunk the time dimension based on your machine to reduce task graph complexity if needed, e.g.:

```python
ds_conus404_sel = ds_conus404_sel.chunk({'time': 72})  # or set a large number like 10000 or -1
ds_aorc_sel = ds_aorc_sel.chunk({'time': 72}) 
```

Now we can load the clipped and time-sliced precipitation data arrays from both gridded datasets, and measure how long it takes to complete this task. Unlike previous steps where only metadata was accessed and the data remained lazily loaded, this operation brings the actual data into memory, which can be slower depending on data size and chunking.

In [ ]:
%%time
ds_conus404_sel_ts = clipped_watershed_datasets["Cottonwood Canyon"]["conus404"].RAINRATE.compute()


In [ ]:
%%time
ds_aorc_sel_ts = clipped_watershed_datasets["Cottonwood Canyon"]["aorc"].RAINRATE.compute()


Check the size of the `time` dimension for both datasets:

In [ ]:
len(ds_conus404_sel_ts.time), len(ds_aorc_sel_ts.time)

In [ ]:
clipped_watershed_datasets

Now that the datasets are fully loaded into memory, we can visualize them directly using Python's `matplotlib` along with `xarray`'s `.plot()` function. Below, we display both precipitation fields for the first time step.

In [ ]:
utils.plot_watershed_data(
    watershed_name = 'Tuolumne River',
    clipped_data_watersheds = clipped_watershed_datasets, # Clipped data contains only xarray Datasets
    gdfs_watersheds = gdfs, # Separate input for GeoDataFrames
    variable_of_interest = 'RAINRATE',
    time_index = 0,
    cmap = 'viridis'
)

utils.plot_watershed_data(
    watershed_name = 'Cottonwood Canyon',
    clipped_data_watersheds = clipped_watershed_datasets, # Clipped data contains only xarray Datasets
    gdfs_watersheds = gdfs, # Separate input for GeoDataFrames
    variable_of_interest = 'RAINRATE',
    time_index = 0,
    cmap = 'viridis'
)

utils.plot_watershed_data(
    watershed_name = 'Ottauquechee River',
    clipped_data_watersheds = clipped_watershed_datasets, # Clipped data contains only xarray Datasets
    gdfs_watersheds = gdfs, # Separate input for GeoDataFrames
    variable_of_interest = 'RAINRATE',
    time_index = 0,
    cmap = 'viridis'
)

In [ ]:
t0_conus = ds_conus404_sel_ts.isel(time=0)
t0_aorc = ds_aorc_sel_ts.isel(time=0)

# Determine common color scale limits
vmin = min(t0_conus.min().item(), t0_conus.min().item())
vmax = max(t0_aorc.max().item(), t0_aorc.max().item())

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, 
                         sharey=True, constrained_layout=True)

# Plot CONUS404
p1 = t0_conus.plot(
    ax=axes[0], vmin=vmin, vmax=vmax, cmap='viridis', add_colorbar=False
)
gdfs["Cottonwood Canyon"].plot(ax=axes[0], facecolor='none', edgecolor='k')
axes[0].set_title(f'CONUS404 Precipitationat time: {pd.to_datetime(ds_conus404_sel_ts.time.values[0])}')

# Plot AORC
p2 = t0_aorc.plot(
    ax=axes[1], vmin=vmin, vmax=vmax, cmap='viridis', add_colorbar=False
)
gdfs["Cottonwood Canyon"].plot(ax=axes[1], facecolor='none', edgecolor='k')
axes[1].set_title(f'AORC Precipitation at time: {pd.to_datetime(ds_aorc_sel_ts.time.values[0])}')

# Shared colorbar
fig.colorbar(p2, ax=axes, orientation='vertical', fraction=0.05, pad=0.02, label='Precipitation (mm/hr)')

plt.show()


In [ ]:
# # Save the clipped, time-sliced, datasets if needed
# ds_conus404_sel_ts.drop_vars("metpy_crs").to_netcdf('./conus404_test.nc')
# ds_aorc_sel_ts.to_netcdf('./aorc_test.nc')


<br></br>
<div style="display: flex; align-items: center; margin-bottom: 0.3em;">
  <div style="display: flex; align-items: center;">
    <span style="color:#004080; font-size: 1.7em; font-weight: bold;">Investigating Biases, Variability, and Correlations</span>
  </div>
</div>

<p>
  In this section, we explore the <strong>statistical agreement and spatial-temporal differences</strong> between the two precipitation datasets by analyzing a suite of key metrics. These metrics are grouped into three broad categories: 
  (1) <strong>central tendency and spread</strong>: including domain-averaged mean, standard deviation (std), and interquartile range (IQR); 
  (2) <em>error-based metrics</em>: such as bias and root mean square error (RMSE); and 
  (3) <em>correlation measures</em>: focusing on pixel-wise anomaly correlation to assess temporal agreement across space. 
  Together, these help us assess whether the two datasets differ systematically (bias), how variable they are over time and space (spread), and how strongly they co-vary (correlation).
</p>

<p>
  To visualize these comparisons, we provide a collection of plots: scatter plots of domain-averaged means to evaluate agreement in bulk statistics; time series with shaded standard deviation envelopes to examine temporal variability; violin plots and histograms to depict distributional differences at specific time steps; bias and RMSE maps to assess spatial error patterns; and a pixel-wise correlation map to show localized temporal similarity. 
</p>


In [ ]:
import importlib
import utils
importlib.reload(utils)

corr = utils.plot_pixelwise_correlation(ds_conus404_sel_ts, 
                                           ds_aorc_sel_ts, 
                                           min_std=1e-6, 
                                           min_valid_obs=2)

# The warning RuntimeWarning: Degrees of freedom <= 0 for 
# slice is not about the global count of valid time steps
# It is about specific pixels that have NaN at every time step after clipping.

This image shows the result of a pixelwise temporal Pearson correlation between two gridded datasets over a specific watershed region. Each grid cell represents a spatial location with a size of 1 km within the watershed. The color indicates the Pearson correlation coefficient (ranging from approximately 0.35 to 0.45 in this example). Higher values (redder shades) indicate stronger agreement in the time series at those locations, while lower values (bluer shades) reflect weaker agreement. Since water flows from east to west in this region, it appears that correlation is lower over higher elevations.

The histogram on the right shows the distribution of correlation values across all pixels. Most values fall in the range of 0.38 to 0.44, indicating moderate but consistent agreement across the domain. No extreme outliers (e.g., negative values or values greater than 0.5) are observed.

This visualization serves as a useful diagnostic for identifying where each dataset may be more or less reliable and whether further bias correction or harmonization is necessary. As a sanity check, we can look at how many valid overlapping time points exist at each pixel, e.g.:

```python
(ds_conus404_sel_ts.notnull() & ds_aorc_sel_ts.notnull()).sum(dim='time').plot()
(ds_conus404_sel_ts.notnull() & ds_aorc_sel_ts.notnull()).sum(dim='time').max()
```

Now, we can compute and plot other spatial statistics for each dataset (e.g., mean, standard deviation):

In [ ]:
mean1 = utils.compute_mean(ds_conus404_sel_ts)
mean2 = utils.compute_mean(ds_aorc_sel_ts)

std1 = utils.compute_std(ds_conus404_sel_ts)
std2 = utils.compute_std(ds_aorc_sel_ts)

iqr1 = utils.compute_iqr(ds_conus404_sel_ts)
iqr2 = utils.compute_iqr(ds_aorc_sel_ts)

utils.plot_spatial_stats(
    mean1, mean2,
    std1, std2,
    iqr1, iqr2)

We can also create correlation plots of the spatial statistics between the AORC and CONUS404 datasets:

In [ ]:
# Plot all three scatter plots
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Mean
axs[0].scatter(mean1, mean2, alpha=0.6, color='purple')
axs[0].plot([mean1.min(), mean1.max()], [mean1.min(), mean1.max()], '--k', label='1:1 line')
axs[0].set_title('Mean Comparison')
axs[0].set_xlabel('CONUS404 Mean')
axs[0].set_ylabel('AORC Mean')
axs[0].legend()

# Std
axs[1].scatter(std1, std2, alpha=0.6, color='green')
axs[1].plot([std1.min(), std1.max()], [std1.min(), std1.max()], '--k', label='1:1 line')
axs[1].set_title('Standard Deviation Comparison')
axs[1].set_xlabel('CONUS404 Std')
axs[1].set_ylabel('AORC Std')
axs[1].legend()

# IQR
axs[2].scatter(iqr1, iqr2, alpha=0.6, color='orange')
axs[2].plot([iqr1.min(), iqr1.max()], [iqr1.min(), iqr1.max()], '--k', label='1:1 line')
axs[2].set_title('IQR Comparison')
axs[2].set_xlabel('CONUS404 IQR')
axs[2].set_ylabel('AORC IQR')
axs[2].legend()

plt.tight_layout()
plt.show()


<br></br>
<div style="display: flex; align-items: center; margin-bottom: 0.3em;">
  <div style="display: flex; align-items: center;">
    <span style="color:#004080; font-size: 1.7em; font-weight: bold;">Saving Analysis Data to HydroShare</span>
  </div>
</div>

Once this is done, we can now save the analysis results back to HydroShare in a programmtic and structured way. First, let's import the relevant libraries:


In [ ]:
import s3fs # providing file-like access to Amazon S3 storage
import requests
from getpass import getpass # making secure prompts for passwords
from fsspec.callbacks import TqdmCallback # tracking file operations with progress bars

We can first convert DataArrays to DataSets:

In [ ]:
ds404 = ds_conus404_sel_ts.to_dataset()
ds_aorc = ds_aorc_sel_ts.to_dataset()

Before saving the dataset, we need to remove metadata that could cause conflicts. `MetPy` automatically adds coordinate reference system (CRS) information, which is stored in the `metpy_crs` variable and referenced through the `grid_mapping` attribute on other variables; both need to be cleared. In addition, because we are changing the chunking scheme for saving to Zarr, any metadata related to the original chunking from when the data was first loaded must be removed to avoid conflicts when writing the new Zarr to disk. 

Let's clean and clean and rechunk the CONUS 404 DataSet:

In [ ]:
ds404_clean = ds404.drop_vars("metpy_crs")

# Remove 'grid_mapping' attribute from all variables (like RAINRATE)
for var in ds404_clean.data_vars:
    ds404_clean[var].attrs.pop("grid_mapping", None)  # safely remove if it exists

# Remove all leftover .encoding entries to avoid chunk metadata conflict
for var in ds404_clean.data_vars:
    ds404_clean[var].encoding.clear()

# re-chunk
ds404_clean = ds404_clean.chunk({'time': 1000, 'y': 40, 'x': 29})

And then clean and rechunk the AORC DataSet:

In [ ]:
# Remove all .encoding entries to avoid chunk metadata conflict
for var in ds_aorc.data_vars:
    ds_aorc[var].encoding.clear()

# re-chunk
ds_aorc = ds_aorc.chunk({'time': 1000, 'y': 40, 'x': 29})

Now we can save the cleaned datasets to Zarr locally:

In [ ]:
ds404_clean.to_zarr("conus404.zarr", mode="w", consolidated=True)
ds_aorc.to_zarr("aorc.zarr", mode="w", consolidated=True)

The next set of operations interface with HydroShare. First, we can upload the Zarr stores to HydroShare:

In [ ]:
_ = s3.put('aorc.zarr', s3_path, recursive=True, callback=TqdmCallback())
_ = s3.put('conus404.zarr', s3_path, recursive=True, callback=TqdmCallback())

Now let's test accessing these data from our HydroShare resource:

In [ ]:

username = " " # <-- Insert Username Here.
password = getpass('Enter your password: ')
resource_id = "3c8c4ef6c5454396b050f30e276da341"

We can connect to the S3 backend that corresponds with our resource:

In [ ]:

# create access key and secret for the authenticated user
response = requests.post(f"https://beta.hydroshare.org/hsapi/user/service/accounts/s3/", auth=(username, password))
key, secret = response.json().values()

response = requests.get(f"https://beta.hydroshare.org/hsapi/resource/s3/{resource_id}/", auth=(username, password))
s3_path = f"{response.json()['bucket']}/{response.json()['prefix']}"

s3 = s3fs.S3FileSystem(key=key,
                       secret=secret,
                       endpoint_url='https://s3.beta.hydroshare.org')

Finally, we can load the AORC Zarr store using `xarray`:

In [ ]:
zarr_path=f"s3:///{s3_path}aorc.zarr"

print(f"Reading AORC from HydroShare...", end='')

store=s3fs.S3Map(
            root=zarr_path,
            s3=s3,
            check=False
        )

ds = xr.open_zarr(
    store=store,
    consolidated=True,
)

ds